# Scanning dataset analysis

In [1]:
import os

import numpy as np
import pandas as pd
import plotly.colors as pcolors
import plotly.express as px
import plotly.graph_objects as go
from Bio.PDB import PDBParser
from scipy.spatial import cKDTree

## 3D PDB structure viewer (plotly)

`plot_structure_3d` loads a `.pdb` file and returns a plotly figure: one marker per atom, one line segment per bond. `ca_only=True` simplifies the view to just each residue's CA atom, connected residue-to-residue along the backbone (a real bond-distance cutoff doesn't apply between CA atoms - they aren't directly bonded to each other).

In [2]:
def load_structure_atoms(pdb_path, chains=None, ca_only=False):
    """
    Parses a PDB file and returns a DataFrame of its atoms - one row per atom (or, if
    ca_only, one row per residue's CA atom only): chain, resseq, resname, atom_name,
    element, x, y, z.

    chains: optional iterable of chain IDs to restrict to (default: every chain).
    """
    parser = PDBParser(QUIET=True)
    structure = parser.get_structure(os.path.basename(pdb_path), pdb_path)
    model = next(structure.get_models())

    rows = []
    for chain in model:
        if chains is not None and chain.id not in chains:
            continue
        for residue in chain:
            for atom in residue:
                if ca_only and atom.get_name() != "CA":
                    continue
                x, y, z = atom.get_coord()
                element = (atom.element or atom.get_name()[0]).strip().upper()
                rows.append({
                    "chain": chain.id,
                    "resseq": residue.id[1],
                    "resname": residue.get_resname(),
                    "atom_name": atom.get_name(),
                    "element": element,
                    "x": float(x), "y": float(y), "z": float(z),
                })
    return pd.DataFrame(rows)

In [3]:
# Standard single-bond covalent radii (Angstroms, Cordero et al. 2008 - common subset).
COVALENT_RADII = {"H": 0.31, "C": 0.76, "N": 0.71, "O": 0.66, "S": 1.05, "P": 1.07}
DEFAULT_COVALENT_RADIUS = 0.77  # fallback for any other element


def infer_bonds(atoms, ca_only=False, tolerance=1.2):
    """
    Returns a list of (i, j) row-index pairs into `atoms` representing bonds to draw.

    Full-atom mode: a pair of atoms is bonded if their distance is within `tolerance` x
    the sum of their elements' covalent radii - a simple, generic, element-aware distance
    cutoff rather than a hardcoded per-residue topology table. A cKDTree keeps this to
    nearby candidate pairs only, not an all-pairs O(n^2) scan.

    CA-only mode: consecutive CA atoms aren't actually bonded to each other (the real
    backbone passes through several intervening atoms), so "bonds" here are a backbone
    trace instead - a line from each residue's CA to the next residue's CA within the
    same chain, only where resseq is truly consecutive (differs by exactly 1). That skip-
    if-not-adjacent check matters whenever residues are missing from `atoms` - an
    unresolved stretch in the original structure, or residues deliberately filtered out
    (e.g. plot_structure_3d's hide_tcr_framework) - so the trace doesn't draw a long
    spurious line straight across the gap. `tolerance` is unused in this mode.
    """
    if ca_only:
        bonds = []
        for _, group in atoms.groupby("chain"):
            sorted_group = group.sort_values("resseq")
            idx = sorted_group.index.to_list()
            resseqs = sorted_group["resseq"].to_list()
            for a, b, ra, rb in zip(idx[:-1], idx[1:], resseqs[:-1], resseqs[1:]):
                if rb - ra == 1:
                    bonds.append((a, b))
        return bonds

    coords = atoms[["x", "y", "z"]].to_numpy()
    radii = atoms["element"].map(COVALENT_RADII).fillna(DEFAULT_COVALENT_RADIUS).to_numpy()

    max_possible_bond = 2 * max(radii.max(), DEFAULT_COVALENT_RADIUS) * tolerance
    tree = cKDTree(coords)
    candidate_pairs = tree.query_pairs(r=max_possible_bond, output_type="ndarray")

    bonds = []
    for i, j in candidate_pairs:
        dist = np.linalg.norm(coords[i] - coords[j])
        if dist <= (radii[i] + radii[j]) * tolerance:
            bonds.append((int(i), int(j)))
    return bonds

### Segment labels (CDR1/2/3, framework, MHC groove helices, peptide)

Rather than inventing a new residue classification, this reuses the project's own per-sample JSON annotation - the same `Sequence[...]['Consensus']['Turn']` IMGT framework/CDR labeling `PrepNetTCR.py` already relies on to extract CDR3, plus the `Region` dict's `MHCha`/`MHChb` groove-helix spans (the same naming already used throughout `data/Features/features.csv`'s `energy:region_energy:*` columns). Chain IDs follow this project's structure files: `A`=MHCa, `C`=Peptide, `D`=TCRa, `E`=TCRb (confirmed by direct coordinate comparison between `TCR.pdb` and `Complex.pdb`).

In [4]:
import json

CHAIN_TO_SEQKEY = {"A": "MHCa", "C": "Peptide", "D": "TCRa", "E": "TCRb"}

SEGMENT_COLORS = {
    "CDR1a": "#6baed6", "CDR2a": "#3182bd", "CDR3a": "#08306b",
    "CDR1b": "#fd8d3c", "CDR2b": "#e6550d", "CDR3b": "#a63603",
    "TCRva": "#bdbdbd", "TCRca": "#636363",
    "TCRvb": "#d9b38c", "TCRcb": "#8c6d31",
    "MHCha": "#41ab5d", "MHChb": "#006d2c", "MHCa": "#a1d99b",
    "Peptide": "#6a51a3",
    "WT": "#808080",
    "DMUT": "#000000",
    "other": "#444444",
}


def infer_json_path(pdb_path):
    """<.../dat/<name>/Structure/*.pdb> -> <.../dat/<name>/<name>.json> - this project's
    fixed per-sample directory convention."""
    structure_dir = os.path.dirname(pdb_path)
    sample_dir = os.path.dirname(structure_dir)
    name = os.path.basename(sample_dir)
    return os.path.join(sample_dir, f"{name}.json")


def load_segment_labels(json_path):
    """
    Returns {chain_id: [segment_label, ...]} in PDBseq/file order, one label per residue.
    """
    with open(json_path) as f:
        data = json.load(f)

    labels = {}
    for chain, seqkey in CHAIN_TO_SEQKEY.items():
        seq = data["Sequence"][seqkey]
        n = len(seq["PDBseq"])
        region = data["Region"].get(seqkey, {})

        if seqkey == "Peptide":
            labels[chain] = ["Peptide"] * n
        elif seqkey == "MHCa":
            chain_labels = ["MHCa"] * n
            for seg in ("MHCha", "MHChb"):
                lo, hi = region[seg]
                for i in range(lo, hi + 1):
                    chain_labels[i] = seg
            labels[chain] = chain_labels
        else:  # TCRa / TCRb
            chain_letter = seqkey[-1]  # 'a' or 'b'
            turn = seq["Consensus"]["Turn"]
            v_lo, v_hi = region[f"TCRv{chain_letter}"]
            chain_labels = []
            for i, t in enumerate(turn):
                if t in ("cdr1", "cdr2", "cdr3"):
                    chain_labels.append(f"{t.upper()}{chain_letter}")
                elif v_lo <= i <= v_hi:
                    chain_labels.append(f"TCRv{chain_letter}")
                else:
                    chain_labels.append(f"TCRc{chain_letter}")
            labels[chain] = chain_labels
    return labels


def assign_segments(atoms, json_path):
    """
    Adds a "segment" column to an atoms DataFrame (from load_structure_atoms), matching
    residues to this structure's own JSON annotation by order of appearance within each
    chain - NOT by resseq value, which doesn't reliably correspond 1:1 to the JSON's own
    residue indexing (e.g. A6_T98's named position is actually residue 92 in both the
    plain sequence index and the PDB resseq - see the mutation-finding section below).
    """
    chain_labels = load_segment_labels(json_path)
    atoms = atoms.copy()
    atoms["segment"] = "other"
    for chain, chain_seg_labels in chain_labels.items():
        chain_mask = atoms["chain"] == chain
        resseqs_in_order = atoms.loc[chain_mask, "resseq"].drop_duplicates().to_list()
        if len(resseqs_in_order) != len(chain_seg_labels):
            raise ValueError(
                f"Chain {chain}: {len(resseqs_in_order)} residues in structure vs "
                f"{len(chain_seg_labels)} in {json_path} - can't assign segments positionally."
            )
        resseq_to_label = dict(zip(resseqs_in_order, chain_seg_labels))
        atoms.loc[chain_mask, "segment"] = atoms.loc[chain_mask, "resseq"].map(resseq_to_label)
    return atoms

In [5]:
QUALITATIVE_PALETTE = pcolors.qualitative.Plotly  # fallback palette for chain/element coloring
TCR_FRAMEWORK_SEGMENTS = {"TCRva", "TCRca", "TCRvb", "TCRcb"}


def build_category_colors(values, color_by, color_map=None):
    """
    Explicit {value: color} map used for BOTH atom markers and bond lines, so they share
    identical colors instead of being assigned independently (bonds used to be a single
    flat grey regardless of color_by, which read as disconnected from the colorful
    atoms). color_map, if given, overrides the auto-assigned color for any category
    present in it (e.g. when the default qualitative palette's first few colors turn out
    too close to each other for a particular structure's chain count).
    """
    if color_by == "segment":
        colors = {v: SEGMENT_COLORS.get(v, "#000000") for v in values}
    else:
        sorted_values = sorted(values)
        colors = {v: QUALITATIVE_PALETTE[i % len(QUALITATIVE_PALETTE)] for i, v in enumerate(sorted_values)}
    if color_map:
        colors.update(color_map)
    return colors


def plot_structure_3d(pdb_path, ca_only=False, chains=None, color_by="chain", bond_tolerance=1.2,
                       marker_size=None, bond_width=None, bond_width_ratio=1.6, highlight_scale=6.0,
                       json_path=None, highlight=None, hide_tcr_framework=False, show_atoms=True,
                       color_map=None, crosslinks=None, crosslink_color="gold", crosslink_width=5,
                       crosslink_colorscale=None, crosslink_min_length=1.0, bond_opacity=1.0, title=None):
    """
    Loads a PDB file and returns a plotly 3D Figure: one marker per atom, one line
    segment per inferred bond (see infer_bonds) - bonds and atoms share the same
    color-by-category mapping (see build_category_colors).

    ca_only: show only each residue's CA atom - a simplified backbone-trace view.
    chains: optional iterable of chain IDs to restrict to (default: every chain).
    color_by: "chain", "element", or "segment" (CDR1/2/3, framework, MHC groove helices,
        peptide - see assign_segments; uses the fixed SEGMENT_COLORS palette).
    color_map: optional {category_value: color} overriding the auto-assigned palette for
        specific categories (e.g. a hand-picked, more distinct set of chain colors).
    bond_tolerance: passed to infer_bonds (ignored in ca_only mode).
    marker_size: atom marker size in px (default: 4 if ca_only else 2.5).
    bond_width: bond line width in px (default: bond_width_ratio x marker_size - bonds
        are drawn thicker than the atoms by default, not thinner).
    show_atoms: if False, skip the atom marker traces entirely - just bonds (and
        crosslinks, if given). Useful to declutter a view down to the backbone trace.
    json_path: this sample's annotation JSON, needed for color_by="segment" and/or
        hide_tcr_framework - inferred from pdb_path's directory layout if not given (see
        infer_json_path).
    highlight: optional iterable of (chain, resseq) tuples - those residues' CA markers
        are drawn much larger (highlight_scale x marker_size) but in the SAME color as
        every other atom in their category - highlighting is purely a size cue, not a
        separate color, so this view and a Kd bar plot colored by the same category line
        up directly.
    hide_tcr_framework: drop TCRva/TCRca/TCRvb/TCRcb residues (everything on the TCR
        chains outside CDR1/2/3) entirely, leaving just the CDR loops (plus MHC/peptide,
        untouched) - an experiment in decluttering the view to just the
        binding-interface-relevant parts of the TCR. Forces segment assignment even if
        color_by isn't "segment".
    crosslinks: optional iterable of ((chain1, resseq1), (chain2, resseq2)) pairs - drawn
        as straight lines between each pair's CA atoms (e.g. for XL-MS crosslink
        validation).
    crosslink_color: flat line color used when crosslink_colorscale is None (the
        default) - every crosslink the same color.
    crosslink_colorscale: if given (a named plotly colorscale, e.g. "Viridis"), each
        crosslink is instead colored individually along this colorscale by its own 3D
        length (CA-CA distance) relative to the other crosslinks being plotted, with a
        colorbar added - a visual cue for which crosslinks are more/less structurally
        plausible (XL-MS crosslinkers typically can't bridge much more than ~30 A).
    crosslink_min_length: floor (in Angstroms) applied to crosslink length before it
        feeds the colorscale normalization - guards against a degenerate near-zero
        length dominating the color range. Ignored unless crosslink_colorscale is given.
    crosslink_width: line width in px for crosslink lines (kept thicker than the
        structure's own bonds by default, so they read as a distinct overlay).
    bond_opacity: opacity (0-1) of the structure's own bond lines - lower this (e.g. 0.3-0.5)
        to fade the backbone so crosslink lines stand out more against it. Does not affect
        crosslink line opacity, which is always fully opaque.
    """
    atoms = load_structure_atoms(pdb_path, chains=chains, ca_only=ca_only)
    if atoms.empty:
        raise ValueError(f"No atoms loaded from {pdb_path} (chains={chains}, ca_only={ca_only})")
    if color_by == "segment" or hide_tcr_framework:
        atoms = assign_segments(atoms, json_path or infer_json_path(pdb_path))
    if hide_tcr_framework:
        atoms = atoms[~atoms["segment"].isin(TCR_FRAMEWORK_SEGMENTS)].reset_index(drop=True)
        if atoms.empty:
            raise ValueError("hide_tcr_framework removed every atom - check chains/color_by inputs")

    bonds = infer_bonds(atoms, ca_only=ca_only, tolerance=bond_tolerance)

    marker_size = marker_size if marker_size is not None else (4 if ca_only else 2.5)
    bond_width = bond_width if bond_width is not None else marker_size * bond_width_ratio

    highlight_set = set(highlight) if highlight else set()
    atoms["is_highlight"] = [
        (c, r) in highlight_set and a == "CA"
        for c, r, a in zip(atoms["chain"], atoms["resseq"], atoms["atom_name"])
    ]

    category_colors = build_category_colors(atoms[color_by].unique(), color_by, color_map=color_map)

    fig = go.Figure()

    # Bonds, grouped and colored by the first atom's category - a bond straddling a
    # segment boundary just takes the earlier residue's color, a minor simplification.
    bond_groups = {}
    for i, j in bonds:
        cat = atoms.at[i, color_by]
        g = bond_groups.setdefault(cat, {"x": [], "y": [], "z": []})
        g["x"] += [atoms.at[i, "x"], atoms.at[j, "x"], None]
        g["y"] += [atoms.at[i, "y"], atoms.at[j, "y"], None]
        g["z"] += [atoms.at[i, "z"], atoms.at[j, "z"], None]
    for cat, g in bond_groups.items():
        fig.add_trace(go.Scatter3d(
            x=g["x"], y=g["y"], z=g["z"],
            mode="lines", line=dict(color=category_colors[cat], width=bond_width),
            opacity=bond_opacity, hoverinfo="skip", showlegend=False,
        ))

    if show_atoms:
        for value in sorted(atoms[color_by].unique()):
            subset = atoms[atoms[color_by] == value]
            sizes = np.where(subset["is_highlight"], marker_size * highlight_scale, marker_size)
            fig.add_trace(go.Scatter3d(
                x=subset["x"], y=subset["y"], z=subset["z"],
                mode="markers",
                marker=dict(size=sizes, color=category_colors[value], opacity=1.0),
                name=str(value),
                text=[
                    f"{row.chain} {row.resname}{row.resseq} {row.atom_name}"
                    for row in subset.itertuples()
                ],
                hoverinfo="text",
            ))

    if crosslinks:
        ca_atoms = atoms if ca_only else load_structure_atoms(pdb_path, chains=chains, ca_only=True)
        ca_pos = {(r.chain, r.resseq): np.array([r.x, r.y, r.z]) for r in ca_atoms.itertuples()}

        resolved = []
        missing = []
        for (c1, r1), (c2, r2) in crosslinks:
            if (c1, r1) not in ca_pos or (c2, r2) not in ca_pos:
                missing.append(((c1, r1), (c2, r2)))
                continue
            p1, p2 = ca_pos[(c1, r1)], ca_pos[(c2, r2)]
            resolved.append((p1, p2, float(np.linalg.norm(p1 - p2))))
        if missing:
            print(f"Skipped {len(missing)} crosslink(s) referencing a residue not in this structure: {missing[:5]}{' ...' if len(missing) > 5 else ''}")

        if resolved and crosslink_colorscale:
            lengths = np.array([max(length, crosslink_min_length) for _, _, length in resolved])
            lo, hi = float(lengths.min()), float(lengths.max())
            norm = ((lengths - lo) / (hi - lo)) if hi > lo else np.zeros_like(lengths)
            link_colors = pcolors.sample_colorscale(crosslink_colorscale, norm.tolist())
            for (p1, p2, length), color in zip(resolved, link_colors):
                fig.add_trace(go.Scatter3d(
                    x=[p1[0], p2[0]], y=[p1[1], p2[1]], z=[p1[2], p2[2]],
                    mode="lines", line=dict(color=color, width=crosslink_width),
                    name="Crosslink", showlegend=False,
                    hovertext=f"{length:.1f} Å", hoverinfo="text",
                ))
            # Invisible marker trace purely to render a shared colorbar for the
            # per-line colors above (a plain "lines" trace can't drive a colorbar).
            fig.add_trace(go.Scatter3d(
                x=[None], y=[None], z=[None], mode="markers",
                marker=dict(
                    size=0.0001, color=[lo, hi], colorscale=crosslink_colorscale,
                    cmin=lo, cmax=hi, showscale=True,
                    colorbar=dict(title="Crosslink<br>length (Å)", x=1.02),
                ),
                showlegend=False, hoverinfo="skip",
            ))
        elif resolved:
            link_x, link_y, link_z = [], [], []
            for p1, p2, _ in resolved:
                link_x += [p1[0], p2[0], None]
                link_y += [p1[1], p2[1], None]
                link_z += [p1[2], p2[2], None]
            fig.add_trace(go.Scatter3d(
                x=link_x, y=link_y, z=link_z,
                mode="lines", line=dict(color=crosslink_color, width=crosslink_width),
                name="Crosslink", hoverinfo="skip",
            ))

    # aspectmode="data" keeps the box true to the structure's actual relative dimensions
    # (rather than forcing a cube); dragmode="orbit" is what actually makes free rotation
    # work - plotly doesn't always default to it depending on the environment.
    fig.update_layout(
        title=title or os.path.basename(pdb_path),
        scene=dict(
            aspectmode="data", xaxis_title="x", yaxis_title="y", zaxis_title="z",
            dragmode="orbit",
        ),
        legend_title=color_by,
        width=800, height=700,
    )
    return fig

### Demo - a scan sample's TCR structure

In [6]:
# A6_WT is the wild-type TCR the scanning dataset's point mutants (A6_R27, A6_T98, ...) are built from.
sample_pdb = "../dat/A6_WT/Structure/TCR.pdb"

plot_structure_3d(sample_pdb, ca_only=True).show()

In [7]:
# Full-atom view, with bonds inferred from covalent-radius distances - slower to render
# and rotate than ca_only, so best for a single structure at a time rather than many.
plot_structure_3d(sample_pdb, ca_only=False).show()

## Locating scan mutations on the structure

The scan sample names encode a mutation as a single-letter WT residue + a number (e.g. `A6_T98`), but that number is neither the plain sequence position nor the PDB's own `resseq` - e.g. `A6_T98` turns out to be residue 92 in both. These are IMGT-style numbers, not reliable to parse directly. Instead, `find_mutated_residues` diffs the actual modeled structure (WT vs mutant, matched by chain + resseq) to find where the residue identity really changed - structure-grounded, so it doesn't depend on trusting the naming convention at all.

In [8]:
def find_mutated_residues(wt_pdb, mutant_pdb, chains=None):
    """
    Compares two PDB files residue-by-residue (matched by chain + resseq, via each
    structure's CA atoms) and returns the list of (chain, resseq, wt_resname,
    mutant_resname) positions where the residue identity differs.

    Only looks at chains/positions present in BOTH files - a mutation on a chain not
    modeled in `mutant_pdb` simply won't show up here, rather than raising an error.
    """
    wt_atoms = load_structure_atoms(wt_pdb, chains=chains, ca_only=True)
    mut_atoms = load_structure_atoms(mutant_pdb, chains=chains, ca_only=True)
    wt_map = {(r.chain, r.resseq): r.resname for r in wt_atoms.itertuples()}
    mut_map = {(r.chain, r.resseq): r.resname for r in mut_atoms.itertuples()}

    return [
        (chain, resseq, wt_map[(chain, resseq)], mut_resname)
        for (chain, resseq), mut_resname in mut_map.items()
        if (chain, resseq) in wt_map and wt_map[(chain, resseq)] != mut_resname
    ]


def find_scan_mutations(wt_pdb, mutant_names, structure_dir="../dat", structure_file="Complex.pdb"):
    """
    Runs find_mutated_residues for every sample name in `mutant_names` against the shared
    `wt_pdb`, looking up each one's own structure under
    structure_dir/<name>/Structure/<structure_file>. Uses Complex.pdb by default (not
    TCR.pdb), so mutations on the peptide (chain C) or MHC (chain A) are caught too, not
    just the TCR chains (D, E). Returns {sample_name: [(chain, resseq, wt_resname,
    mut_resname), ...]}.
    """
    return {
        name: find_mutated_residues(wt_pdb, f"{structure_dir}/{name}/Structure/{structure_file}")
        for name in mutant_names
    }

In [9]:
annotation = pd.read_csv("../data/ML/Annotation.csv")
scan_names = annotation[annotation["source"] == "scan"]["name"].tolist()

a6_mutants = [n for n in scan_names if n.startswith("A6_") and n != "A6_WT"]
b7_mutants = [n for n in scan_names if n.startswith("B7_") and n != "B7_WT"]

a6_diffs = find_scan_mutations("../dat/A6_WT/Structure/Complex.pdb", a6_mutants)
b7_diffs = find_scan_mutations("../dat/B7_WT/Structure/Complex.pdb", b7_mutants)

for label, diffs in [("A6", a6_diffs), ("B7", b7_diffs)]:
    print(f"--- {label} ---")
    for name, d in diffs.items():
        print(f"  {name}: {d}")

--- A6 ---
  A6_R27: [('D', 27, 'ARG', 'ALA')]
  A6_T98: [('D', 92, 'THR', 'ALA')]
  A6_K68: [('D', 66, 'LYS', 'ALA')]
  A6_S31: [('D', 31, 'SER', 'ALA')]
  A6_E30: [('D', 31, 'SER', 'ALA'), ('E', 30, 'GLU', 'ALA')]
  A6_L98: [('D', 31, 'SER', 'ALA'), ('E', 97, 'LEU', 'ALA')]
  A6_Y5: [('C', 5, 'TYR', 'ALA'), ('D', 31, 'SER', 'ALA')]
  A6_Y8: [('C', 8, 'TYR', 'ALA'), ('D', 31, 'SER', 'ALA')]
  A6_R65: [('A', 65, 'ARG', 'ALA'), ('D', 31, 'SER', 'ALA')]
  A6_T98_R65: [('A', 65, 'ARG', 'ALA'), ('D', 92, 'THR', 'ALA')]
  A6_L98_Y8: [('C', 8, 'TYR', 'ALA'), ('D', 31, 'SER', 'ALA'), ('E', 97, 'LEU', 'ALA')]
  A6_E30_Y8: [('C', 8, 'TYR', 'ALA'), ('D', 31, 'SER', 'ALA'), ('E', 30, 'GLU', 'ALA')]
  A6_S31_Y5: [('C', 5, 'TYR', 'ALA'), ('D', 31, 'SER', 'ALA')]
--- B7 ---
  B7_E30: [('E', 30, 'GLU', 'ALA')]
  B7_D30: [('D', 30, 'ASP', 'ALA')]
  B7_Y104: [('E', 104, 'TYR', 'ALA')]
  B7_E94: [('D', 93, 'GLU', 'ALA')]
  B7_Y5: [('C', 5, 'TYR', 'ALA')]
  B7_Y8: [('C', 8, 'TYR', 'ALA')]
  B7_R65: [('A'

A couple of things worth noting in the printed diffs above before taking the union:
- `R65` resolves on chain A (the MHC) at resseq 65 - it's an MHC contact-residue mutation, not a TCR one, which is why it never showed up when this only diffed `TCR.pdb`.
- `Y5`/`Y8` resolve on chain C (the peptide) - confirming these are peptide mutations, matching the `Peptide` column changes seen in `Annotation.csv`.
- A few A6 samples (`A6_E30`, `A6_L98`, `A6_Y5`, `A6_Y8`, `A6_R65`) also show `('D', 31, 'SER', 'ALA')` alongside their own named mutation - this looks like a shared background difference between the modeling batch used for those particular mutants and the `A6_WT` reference, not a distinct named point mutation. It's harmless for the union below (position 31 is already a real mutation site via `A6_S31`), just don't read every entry in the printed list above as "the one intended mutation" for that sample.

In [10]:
def mutated_positions(diffs):
    """Union of (chain, resseq) across every sample's diff list."""
    return {(chain, resseq) for d in diffs.values() for chain, resseq, _, _ in d}

a6_positions = mutated_positions(a6_diffs)
b7_positions = mutated_positions(b7_diffs)
print("A6 mutated CA positions:", sorted(a6_positions))
print("B7 mutated CA positions:", sorted(b7_positions))

A6 mutated CA positions: [('A', 65), ('C', 5), ('C', 8), ('D', 27), ('D', 31), ('D', 66), ('D', 92), ('E', 30), ('E', 97)]
B7 mutated CA positions: [('A', 65), ('C', 5), ('C', 8), ('D', 30), ('D', 93), ('E', 30), ('E', 104)]


### A6 and B7 wild-type structures, with scanned residues enlarged

In [11]:
# Complex.pdb keeps the exact same chain IDs/resseq numbering for the TCR chains (D, E)
# as TCR.pdb - confirmed by direct comparison - so the highlight positions found above
# carry over unchanged; it just adds the MHC (chain A) and peptide (chain C) for context.
plot_structure_3d(
    "../dat/A6_WT/Structure/Complex.pdb", ca_only=True, color_by="segment",
    highlight=a6_positions, title="A6_WT complex - scanned residues highlighted",
).show()

In [12]:
plot_structure_3d(
    "../dat/B7_WT/Structure/Complex.pdb", ca_only=True, color_by="segment",
    highlight=b7_positions, title="B7_WT complex - scanned residues highlighted",
).show()

## Calculated Kd per mutation

`data/Scanning/scan_meta.csv`'s `MUT_affinity` column is the experimentally-derived binding free energy (ΔG, kcal/mol) for each scan sample (its `Chain`/`Position` columns also independently confirm the structure-diff mutation finding above - e.g. `A6_T98` is listed there as position "98 (92)", exactly the residue 92 found by diffing). Kd is derived from it via ΔG = RT·ln(Kd), i.e. `Kd = exp(ΔG / RT)` with R=0.0019872 kcal/(mol·K), T=298.15K - this is **the experimental Kd that actually gets plotted**.

This project's own computed `Gbind` (from `data/Features/features.csv`, see `src/Calc_Energy.py`) and its sibling total-energy terms (`Gcom`, `Gpmhc`, `Gtcr`) are kept in the table alongside it for reference/comparison, but aren't what's plotted.

Each bar is colored by its primary mutated segment (the same `SEGMENT_COLORS` used on the structure plots above), so a bar's color tells you directly which structural region that mutation hit.

In [13]:
import re

R_GAS = 0.0019872  # kcal / (mol . K)
TEMPERATURE_K = 298.15

features = pd.read_csv("../data/Features/features.csv")
gbind = features.set_index("name")["energy:total_energy:Gbind"]
OTHER_ENERGY_TERMS = ["energy:total_energy:Gcom", "energy:total_energy:Gpmhc", "energy:total_energy:Gtcr"]
other_energies = features.set_index("name")[OTHER_ENERGY_TERMS]

scan_meta = pd.read_csv("../data/Scanning/scan_meta.csv").dropna(subset=["Name"]).set_index("Name")


def segment_lookup(wt_pdb, json_path=None):
    """{(chain, resseq): segment_label} for every residue in wt_pdb."""
    json_path = json_path or infer_json_path(wt_pdb)
    atoms = assign_segments(load_structure_atoms(wt_pdb, ca_only=True), json_path)
    return {(row.chain, row.resseq): row.segment for row in atoms.itertuples()}


def parse_scan_meta_position(position):
    """
    scan_meta.csv's Position column is either a plain number ("27") or "name (actual)"
    when the IMGT-style naming number differs from the real PDB resseq (e.g. "98 (92)"
    for A6_T98, which really sits at residue 92) - always returns the real resseq as int.
    """
    match = re.search(r"\((\d+)\)", str(position))
    return int(match.group(1)) if match else int(position)


def component_mutation_segments(name, wt_name, scan_meta, lookup):
    """
    Resolves a scan sample's mutated segment(s) directly from scan_meta.csv's own
    Chain/Position columns - authoritative, and avoids relying on structure-diffing
    (find_scan_mutations), which for a handful of A6 mutants also picks up an unrelated
    modeling-batch difference at ('D', 31) that isn't a real named mutation, incorrectly
    making single mutants look like they hit two segments. A combined mutant's name
    decomposes into its individual single-mutation component names (e.g.
    "A6_T98_R65" -> "A6_T98", "A6_R65"), each of which has its own Chain/Position row in
    scan_meta.csv.
    """
    if name == wt_name:
        return ["WT"]
    prefix = wt_name.split("_")[0] + "_"
    codes = name[len(prefix):].split("_")
    segments = []
    for code in codes:
        row = scan_meta.loc[prefix + code]
        resseq = parse_scan_meta_position(row["Position"])
        segments.append(lookup.get((row["Chain"], resseq), "?"))
    return segments


def build_kd_table(wt_name, mutant_names, lookup, gbind, other_energies, scan_meta,
                    R=R_GAS, T=TEMPERATURE_K):
    """
    One row per sample (WT + every mutant): the experimental dG (scan_meta.csv's
    MUT_affinity) and the Kd derived from it (Kd = exp(dG / RT)) - this is what gets
    plotted. This project's own computed Gbind and its sibling total_energy terms are
    also kept, for reference/comparison, not for the plot. MutatedSegment/PrimarySegment
    come from scan_meta's own Chain/Position (see component_mutation_segments), matched
    against `lookup` (segment_lookup's (chain, resseq) -> segment map). `Label` strips
    the shared "<TCR>_" prefix for a cleaner x-axis.
    """
    names = [wt_name] + list(mutant_names)
    df = pd.DataFrame({"name": names})
    df["Gbind"] = df["name"].map(gbind)
    for col in other_energies.columns:
        df[col] = df["name"].map(other_energies[col])
    df["WT_affinity"] = df["name"].map(scan_meta["WT_affinity"])
    df["MUT_affinity"] = df["name"].map(scan_meta["MUT_affinity"])
    df["Kd"] = np.exp(df["MUT_affinity"] / (R * T))

    segments = df["name"].map(lambda n: component_mutation_segments(n, wt_name, scan_meta, lookup))
    df["MutatedSegment"] = segments.map(lambda segs: "+".join(sorted(set(segs))))
    df["PrimarySegment"] = df["name"].map(
        lambda n: "WT" if n == wt_name
        else "DMUT" if scan_meta.loc[n, "Status"] == "DMUT"
        else component_mutation_segments(n, wt_name, scan_meta, lookup)[0]
    )

    prefix = wt_name.split("_")[0] + "_"
    df["Label"] = df["name"].str[len(prefix):]
    return df.sort_values("Kd").reset_index(drop=True)


a6_segment_lookup = segment_lookup("../dat/A6_WT/Structure/Complex.pdb")
b7_segment_lookup = segment_lookup("../dat/B7_WT/Structure/Complex.pdb")

a6_kd = build_kd_table("A6_WT", a6_mutants, a6_segment_lookup, gbind, other_energies, scan_meta)
b7_kd = build_kd_table("B7_WT", b7_mutants, b7_segment_lookup, gbind, other_energies, scan_meta)
a6_kd

,name,Gbind,energy:total_energy:Gcom,energy:total_energy:Gpmhc,energy:total_energy:Gtcr,WT_affinity,MUT_affinity,Kd,MutatedSegment,PrimarySegment,Label
0,A6_K68,-29.039195,-860.399300,-304.493935,-526.866169,-7.810,-9.390,1.309385e-07,TCRva,TCRva,K68
1,A6_WT,-9.769686,-850.052600,-271.568861,-568.714053,-7.715,-7.715,2.212349e-06,WT,WT,WT
2,A6_R27,-12.503992,-946.574192,-310.694334,-623.375866,-7.715,-7.243,4.907189e-06,CDR1a,CDR1a,R27
3,A6_S31,-16.055355,-970.008365,-337.598842,-616.354168,-7.641,-6.620,1.404417e-05,CDR1a,CDR1a,S31
4,A6_Y5,-38.251567,-953.959799,-302.958007,-612.750225,-7.641,-6.109,3.327073e-05,Peptide,Peptide,Y5
5,A6_S31_Y5,-33.161828,-955.780274,-318.909935,-603.708512,-7.641,-5.932,4.485434e-05,CDR1a+Peptide,DMUT,S31_Y5
6,A6_T98,-34.553330,-565.966080,-189.456562,-341.956188,-7.601,-5.656,7.146845e-05,CDR3a,CDR3a,T98
7,A6_L98,-12.970294,-954.679609,-309.670296,-632.039019,-8.267,-5.651,7.207412e-05,CDR3b,CDR3b,L98
8,A6_E30,-19.112387,-994.832870,-328.526245,-647.194238,-7.523,-5.502,9.268242e-05,CDR1b,CDR1b,E30
9,A6_T98_R65,-32.555548,-847.907324,-296.182913,-519.168864,-7.601,-5.179,1.598669e-04,CDR3a+MHCha,DMUT,T98_R65


In [14]:
# Fixed y-range/ticks shared by every Kd plot, rather than each one auto-ranging to its
# own data - makes the two plots directly comparable at a glance. In uM (Kd is computed
# in M, so x1e6).
KD_YMIN_UM = 1e-3   # 1 nM
KD_YMAX_UM = 2e5    # 200 mM
LABEL_FONT_SIZE = 12


def plot_kd_bar(df, title):
    colors = [SEGMENT_COLORS.get(seg, "#000000") for seg in df["PrimarySegment"]]
    kd_uM = df["Kd"] * 1e6
    # Line-break multi-segment labels (e.g. "CDR1a+Peptide" -> "CDR1a+<br>Peptide")
    # instead of letting plotly auto-shrink the text to fit on one line - that auto-
    # shrink is also what was making the font size look inconsistent bar to bar despite
    # a fixed textfont size; constraintext="none" below disables it outright.
    bar_text = df["MutatedSegment"].str.replace("+", "+<br>")
    fig = go.Figure(go.Bar(
        x=df["Label"], y=kd_uM,
        marker_color=colors,
        text=bar_text, textposition="outside", constraintext="none",
        textfont=dict(size=LABEL_FONT_SIZE),
        hovertext=[
            f"{n}: Kd={kd:.3g} uM, MUT_affinity={a:.2f} kcal/mol, Gbind={g:.1f}, segment={seg}"
            for n, kd, a, g, seg in zip(df["name"], kd_uM, df["MUT_affinity"], df["Gbind"], df["MutatedSegment"])
        ],
        hoverinfo="text",
    ))
    # Reference lines at 1 uM and 100 uM, to match the convention used on another plot -
    # thin and partly transparent so they read as background guides, not data.
    fig.add_hline(y=1, line_dash="dash", line_color="black", line_width=1, opacity=0.4)
    fig.add_hline(y=100, line_dash="dash", line_color="black", line_width=1, opacity=0.4)
    fig.update_layout(
        title=title, xaxis_title="Mutation", yaxis_title="Experimental Kd (uM)",
        yaxis=dict(
            type="log", range=[np.log10(KD_YMIN_UM), np.log10(KD_YMAX_UM)], dtick=1,
            exponentformat="none", tickfont=dict(size=LABEL_FONT_SIZE),
        ),
        xaxis=dict(tickangle=-45, tickfont=dict(size=LABEL_FONT_SIZE)),
        showlegend=False, width=500, height=500,
    )
    return fig


plot_kd_bar(a6_kd, "A6 scan - calculated Kd per mutation").show()

In [15]:
plot_kd_bar(b7_kd, "B7 scan - calculated Kd per mutation").show()

## Model predictions vs. experimental Kd, across all scan mutants

One point per (mutant, model): y = that mutant's real experimental Kd (same value regardless of model - it's a property of the mutant, not a prediction), x = model, colored by whether that model predicted it a binder or non-binder. Accuracy/correctness isn't the point here (the scan labels used to train these models are a coarse binder/non-binder call, somewhat arbitrary for point mutants) - this is about whether each model's calls track the real Kd at all, visually.

ML and ResNet both come from 5-fold CV runs scored against the same fixed scan set each fold - only fold 1 is used here (one point per mutant per model, not five). NetTCR and ERGO are each a single fit already, so there's nothing to pick.

In [16]:
ML_MODEL_ABBREV = {
    "Logistic Regression": "LR", "Random Forest": "RF", "Gradient Boosting": "GB",
    "Support Vector Machine": "SVM", "K-Nearest Neighbors": "KNN", "Multilayer Perceptron": "MLP",
}

# Every raw results file here is already in the exact same order as this - confirmed by
# direct row-for-row comparison (CDR3a/CDR3b/peptide) against NetTCR's and ERGO's own
# inputs - so it's the one place "name" gets attached to the id-less ERGO/NetTCR outputs.
scan_names_ordered = annotation[annotation["source"] == "scan"]["name"].reset_index(drop=True)


def load_ml_scan():
    # All 5 CV folds, not just fold 1 - each fold is a separately-trained model scored
    # against the same fixed scan set, so the spread across folds shows real
    # fold-to-fold variance rather than a single arbitrary pick.
    df = pd.read_csv("../data/ML/Results/Base/all_to_scan/All/test_predictions.csv")
    df = df.replace({"Model": ML_MODEL_ABBREV})
    return df.rename(columns={"Probability": "Score"})[["name", "Model", "Fold", "Score"]]


def load_resnet_scan():
    # Same idea as load_ml_scan: all 5 fold files, not just Fold0.
    dfs = []
    for i in range(5):
        d = pd.read_csv(f"../data/DL/Scanning/Fold{i}.csv")
        d = d[d["source"] == "scan"].rename(columns={"Name": "name", "Prob": "Score"}).copy()
        d["Model"] = "ResNet"
        d["Fold"] = i + 1
        dfs.append(d[["name", "Model", "Fold", "Score"]])
    return pd.concat(dfs, ignore_index=True)


def load_nettcr_scan(comparison, model_label):
    preds = pd.read_csv(f"../data/NetTCR/Comparisons/{comparison}/predictions.csv")
    if len(preds) != len(scan_names_ordered):
        raise ValueError(f"{comparison}: {len(preds)} predictions vs {len(scan_names_ordered)} scan samples")
    return pd.DataFrame({"name": scan_names_ordered, "Model": model_label, "Fold": 1, "Score": preds["prediction"]})


def load_ergo_scan(results_file, model_label):
    preds = pd.read_csv(f"../data/ERGO/{results_file}")
    if len(preds) != len(scan_names_ordered):
        raise ValueError(f"{results_file}: {len(preds)} predictions vs {len(scan_names_ordered)} scan samples")
    return pd.DataFrame({"name": scan_names_ordered, "Model": model_label, "Fold": 1, "Score": preds["Score"]})


ml_scan = load_ml_scan()
resnet_scan = load_resnet_scan()

# Soft-voting ensemble of the ML+DL models only (same convention as the FullRun
# ensemble earlier in this notebook: NetTCR/ERGO excluded from the vote itself) -
# averaged across every fold now available, not just a single fold pairing (ML's and
# ResNet's folds come from two independent CV runs, so there's no real "fold 3 of ML
# matches fold 3 of ResNet" pairing to exploit - this just pools everything into one
# robust per-mutant estimate, still a single Ensemble point).
ensemble_scores = (
    pd.concat([ml_scan[["name", "Score"]], resnet_scan[["name", "Score"]]])
    .groupby("name")["Score"].mean()
)
ensemble_scan = pd.DataFrame({"name": ensemble_scores.index, "Model": "Ensemble", "Fold": 1, "Score": ensemble_scores.values})

scan_predictions = pd.concat([
    ml_scan,
    resnet_scan,
    load_nettcr_scan("all_vs_scan", "NetTCR (ours)"),
    load_nettcr_scan("paper_vs_scan", "NetTCR (paper)"),
    load_ergo_scan("scanAE_results.csv", "ERGO-Auto"),
    load_ergo_scan("scanLSTM_results.csv", "ERGO-LSTM"),
    ensemble_scan,
], ignore_index=True)

# Pred is derived the same way for every model/fold here - thresholding Score at 0.5 -
# rather than ever using a model's own native hard-label output. They aren't always the
# same thing: sklearn's SVC in particular fits .predict_proba() via a separate internal
# CV (Platt scaling) that isn't constrained to agree with .predict()'s raw decision-
# boundary side, and with class_weight="balanced" it visibly doesn't for several scan
# mutants (Probability as low as 0.047 with Predicted==1). Thresholding Score uniformly
# keeps the plotted color always consistent with the plotted x-axis/y-axis probability.
scan_predictions["Pred"] = (scan_predictions["Score"] >= 0.5).astype(int)

# scan_meta was already loaded earlier (Kd-per-mutation section) for just A6/B7
# individually - reused here as-is, covering every scan sample at once.
scan_predictions["Kd_uM"] = scan_predictions["name"].map(
    np.exp(scan_meta["MUT_affinity"] / (R_GAS * TEMPERATURE_K)) * 1e6
)

scan_predictions.head()

,name,Model,Fold,Score,Pred,Kd_uM
0,A6_WT,LR,1,0.821853,1,2.212349
1,A6_R27,LR,1,0.834836,1,4.907189
2,A6_T98,LR,1,0.913610,1,71.468447
3,A6_K68,LR,1,0.962435,1,0.130939
4,A6_S31,LR,1,0.895871,1,14.044165


In [17]:
PRED_COLORS = {1: "#2ca02c", 0: "#d62728"}  # predicted binder (green) / non-binder (red)
PRED_LABELS = {1: "Predicted binder", 0: "Predicted non-binder"}
SCAN_MODEL_ORDER = list(ML_MODEL_ABBREV.values()) + [
    "ResNet", "Ensemble", "NetTCR (ours)", "NetTCR (paper)", "ERGO-Auto", "ERGO-LSTM"
]
JITTER_WIDTH = 0.3  # wider than before - ML/ResNet columns are now 5x denser (every fold)
JITTER_SEED = 42

# Categorical x positions jittered by hand (plotly has no built-in jitter for a scatter's
# categorical axis) - map each model to an integer slot, add uniform random noise to it
# for the actual plotted x, then point the tick labels back at the un-jittered integers.
model_slot = {model: i for i, model in enumerate(SCAN_MODEL_ORDER)}
rng = np.random.default_rng(JITTER_SEED)
scan_predictions["x_jittered"] = (
    scan_predictions["Model"].map(model_slot)
    + rng.uniform(-JITTER_WIDTH, JITTER_WIDTH, size=len(scan_predictions))
)

fig = go.Figure()
for pred_value, color in PRED_COLORS.items():
    subset = scan_predictions[scan_predictions["Pred"] == pred_value]
    fig.add_trace(go.Scatter(
        x=subset["x_jittered"], y=subset["Kd_uM"],
        mode="markers",
        marker=dict(color=color, size=7, opacity=0.6, line=dict(color="black", width=0.5)),
        name=PRED_LABELS[pred_value],
        text=subset["name"],
        customdata=subset["Fold"],
        hovertemplate="%{text} (fold %{customdata})<br>Kd=%{y:.3g} uM<extra></extra>",
    ))

fig.update_layout(
    title="Scan mutants: experimental Kd vs. model prediction (all CV folds)",
    xaxis=dict(
        tickmode="array", tickvals=list(model_slot.values()), ticktext=list(model_slot.keys()),
        title="Model", tickangle=-45,
    ),
    yaxis=dict(type="log", title="Experimental Kd (uM)", exponentformat="none"),
    width=1100, height=600,
)
fig.show()

## Experimental Kd vs. predicted probability, faceted per model

Same underlying data as the plot above, viewed differently: each model gets its own panel, x = that model's predicted probability (continuous, not thresholded), y = the mutant's real experimental Kd. A model that's actually tracking real affinity should show probability trending down as Kd goes up; a model with no real signal should look like a random scatter. Colored by the same predicted-class call as before, just so the 0.5 threshold is still visible on top of the continuous trend.

In [18]:
scan_predictions["PredLabel"] = scan_predictions["Pred"].map(PRED_LABELS)

fig = px.scatter(
    scan_predictions, x="Score", y="Kd_uM", facet_col="Model", facet_col_wrap=4,
    category_orders={"Model": SCAN_MODEL_ORDER},
    color="PredLabel", color_discrete_map={PRED_LABELS[1]: PRED_COLORS[1], PRED_LABELS[0]: PRED_COLORS[0]},
    log_y=True, hover_data=["name", "Fold"],
    labels={"Score": "Predicted probability", "Kd_uM": "Experimental Kd (uM)"},
)
fig.update_traces(marker=dict(size=6, opacity=0.6, line=dict(color="black", width=0.5)))
fig.update_yaxes(exponentformat="none")
fig.update_xaxes(range=[-0.05, 1.05])
fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))  # "Model=LR" -> "LR"
fig.update_layout(
    title="Scan mutants: experimental Kd vs. predicted probability, per model (all CV folds)",
    legend_title="", width=1300, height=750,
)
fig.show()

## Physics-term correlations with Kd

Checks every "Phys" feature (`data/Features/features.csv`, columns 364 onward - the same All/Seq/Str/Phys split `TCRclassifier.py`'s `__main__` uses) for a correlation with the scan mutants' experimental Kd. Correlated against log10(Kd) rather than raw Kd, since Kd spans several orders of magnitude (consistent with every Kd axis being log-scaled elsewhere in this notebook) - a linear correlation against the raw value would mostly just reflect the few largest outliers. Both Pearson (linear) and Spearman (rank, more robust to outliers/non-linearity) are reported. With only 26 scan samples, treat this as a leads-worth-checking-further list, not a result: at n=26, |r| needs to be roughly above ~0.39 just to reach p<0.05, and with ~350 physics features tested at once, a few will clear that by chance alone (no multiple-testing correction applied here).

In [19]:
from scipy.stats import pearsonr, spearmanr

features = pd.read_csv("../data/Features/features.csv")
feature_columns = [c for c in features.columns if c != "name"]
# Same column-index convention TCRclassifier.py's __main__ uses to split features into
# All/Seq/Str/Phys (base + sequence[1:164], structural[164:364], physics[364:]) - reused
# here rather than guessing a different boundary.
physics_columns = feature_columns[364:]

scan_features = features[features["name"].isin(scan_meta.index)].set_index("name")
log_kd = np.log10(np.exp(scan_meta["MUT_affinity"] / (R_GAS * TEMPERATURE_K)) * 1e6)  # log10(Kd, uM)
log_kd = log_kd.reindex(scan_features.index)

rows = []
for col in physics_columns:
    values = pd.to_numeric(scan_features[col], errors="coerce")
    if values.isna().all():
        values = values.fillna(0)
    else:
        values = values.fillna(values.median())
    if values.std() == 0:
        continue
    pearson_r, pearson_p = pearsonr(values, log_kd)
    spearman_r, spearman_p = spearmanr(values, log_kd)
    rows.append({
        "Feature": col, "Pearson_r": pearson_r, "Pearson_p": pearson_p,
        "Spearman_r": spearman_r, "Spearman_p": spearman_p,
    })

physics_kd_corr = pd.DataFrame(rows)
physics_kd_corr["AbsPearson_r"] = physics_kd_corr["Pearson_r"].abs()
physics_kd_corr = physics_kd_corr.sort_values("AbsPearson_r", ascending=False).drop(columns="AbsPearson_r")
physics_kd_corr.to_csv("../data/Results/scan_physics_kd_correlation.csv", index=False)

print(f"n={len(scan_features)} scan samples, {len(physics_kd_corr)} physics features tested (of {len(physics_columns)} total)")
physics_kd_corr.head(20)

n=26 scan samples, 246 physics features tested (of 350 total)


,Feature,Pearson_r,Pearson_p,Spearman_r,Spearman_p
15,aro_charge_distance:Peptide_MHChb:min,0.572001,0.002265,0.531031,0.005250
22,aro_charge_distance:Peptide_CDR3a:max,-0.555408,0.003225,-0.528637,0.005497
16,aro_charge_distance:Peptide_MHChb:max,-0.552944,0.003393,-0.382971,0.053472
13,aro_charge_distance:Peptide_MHCha:max,-0.550382,0.003576,-0.366216,0.065765
12,aro_charge_distance:Peptide_MHCha:min,0.529269,0.005431,0.292016,0.147747
23,aro_charge_distance:Peptide_CDR3a:mean,-0.502900,0.008830,-0.518379,0.006668
17,aro_charge_distance:Peptide_MHChb:mean,-0.485816,0.011865,-0.300906,0.135245
173,energy:region_energy:TCRva:bound,-0.455897,0.019248,-0.372029,0.061275
175,energy:region_energy:TCRva:difference,-0.441846,0.023826,-0.396649,0.044834
172,energy:region_energy:TCRa:difference,-0.441846,0.023826,-0.396649,0.044834


### Plotting the significant ones

Every physics feature with Pearson p<0.05 against log10(Kd), one panel each, with an OLS trendline. Plotted against log10(Kd) directly (linear axis) rather than Kd on a log-scaled axis, so the trendline shown is the same fit the correlation table is actually reporting - fitting OLS against raw Kd and only log-scaling the axis for display would visually bend the "line" and no longer match the reported r. Two features (`energy:region_energy:TCRva:difference` and `energy:region_energy:TCRa:difference`) had identical r/p - perfectly collinear in this sample, so only one is shown.

In [20]:
SIG_ALPHA = 0.05
significant_features = (
    physics_kd_corr[physics_kd_corr["Pearson_p"] < SIG_ALPHA]
    .drop_duplicates(subset=["Pearson_r", "Pearson_p"])  # perfectly-collinear duplicates
    ["Feature"].tolist()
)
print(f"{len(significant_features)} features significant at p<{SIG_ALPHA}:")
print(significant_features)

plot_df = scan_features[significant_features].apply(lambda s: pd.to_numeric(s, errors="coerce").fillna(pd.to_numeric(s, errors="coerce").median()))
plot_df["log10_Kd_uM"] = log_kd.reindex(plot_df.index)
plot_df["name"] = plot_df.index

long_df = plot_df.melt(id_vars=["name", "log10_Kd_uM"], value_vars=significant_features, var_name="Feature", value_name="Value")

fig = px.scatter(
    long_df, x="Value", y="log10_Kd_uM", facet_col="Feature", facet_col_wrap=4,
    trendline="ols", hover_data=["name"],
    labels={"log10_Kd_uM": "log10(Experimental Kd, uM)"},
)
fig.update_traces(marker=dict(size=7, opacity=0.75, line=dict(color="black", width=0.5)), selector=dict(mode="markers"))
fig.update_xaxes(matches=None)
fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1], font=dict(size=9)))
fig.update_layout(
    title=f"Scan: physics features significantly correlated with Kd (Pearson p<{SIG_ALPHA})",
    width=1400, height=1000,
)
fig.show()

17 features significant at p<0.05:
['aro_charge_distance:Peptide_MHChb:min', 'aro_charge_distance:Peptide_CDR3a:max', 'aro_charge_distance:Peptide_MHChb:max', 'aro_charge_distance:Peptide_MHCha:max', 'aro_charge_distance:Peptide_MHCha:min', 'aro_charge_distance:Peptide_CDR3a:mean', 'aro_charge_distance:Peptide_MHChb:mean', 'energy:region_energy:TCRva:bound', 'energy:region_energy:TCRva:difference', 'energy:region_energy:TCRa:difference', 'energy:interaction_energy:Peptide_MHChb:max', 'energy:region_energy:TCRva:unbound', 'aro_charge_distance:Peptide_MHCha:mean', 'proximity:CDR3a_CDR3b_20', 'charged_distance:MHCha_CDR1b:mean', 'proximity:Peptide_MHCha_10', 'clashes:MHChb']


## Experimental Kd vs. percent of folds calling it a binder

x = the percentage of that model's folds/runs that called a given mutant a binder (Score >= 0.5), y = experimental Kd. For a single-run model (NetTCR, ERGO), this necessarily collapses to 0% or 100% - there's only one vote. For Ensemble, this pools every individual ML+ResNet fold's own vote directly (6 models x 5 folds + 5 ResNet folds = 35 votes per mutant) and takes the percent of those 35 that were >=0.5 - not the mean probability thresholded afterward, so it's a hard-vote percentage rather than the earlier soft-voting Ensemble score.

In [21]:
# Non-ensemble models: percent of that model's own rows (folds, or the single run) with
# Pred==1 - trivially 0 or 100 for the single-fit models (NetTCR, ERGO), a real
# fold-to-fold percentage for ML/ResNet.
non_ensemble = scan_predictions[scan_predictions["Model"] != "Ensemble"]
percent_binder = (
    non_ensemble.groupby(["name", "Model"])["Pred"].mean().mul(100)
    .rename("PercentFolds").reset_index()
)

# Ensemble: pool every individual ML+ResNet fold's own vote (not the pre-averaged
# Ensemble Score row above) and take the percent of those votes that were >=0.5.
ensemble_percent = (
    pd.concat([ml_scan, resnet_scan])
    .assign(Pred=lambda d: (d["Score"] >= 0.5).astype(int))
    .groupby("name")["Pred"].mean().mul(100)
    .rename("PercentFolds").reset_index()
)
ensemble_percent["Model"] = "Ensemble"

percent_binder = pd.concat([percent_binder, ensemble_percent], ignore_index=True)
percent_binder["Kd_uM"] = percent_binder["name"].map(
    np.exp(scan_meta["MUT_affinity"] / (R_GAS * TEMPERATURE_K)) * 1e6
)
percent_binder["PredLabel"] = (percent_binder["PercentFolds"] >= 50).astype(int).map(PRED_LABELS)

fig = px.scatter(
    percent_binder, x="PercentFolds", y="Kd_uM", facet_col="Model", facet_col_wrap=4,
    category_orders={"Model": SCAN_MODEL_ORDER},
    color="PredLabel", color_discrete_map={PRED_LABELS[1]: PRED_COLORS[1], PRED_LABELS[0]: PRED_COLORS[0]},
    log_y=True, hover_data=["name"],
    labels={"PercentFolds": "% folds predicted binder", "Kd_uM": "Experimental Kd (uM)"},
)
fig.update_traces(marker=dict(size=7, opacity=0.75, line=dict(color="black", width=0.5)))
fig.update_yaxes(exponentformat="none")
fig.update_xaxes(range=[-5, 105])
fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))
fig.update_layout(
    title="Scan mutants: experimental Kd vs. % of folds predicting binder, per model",
    legend_title="", width=1300, height=750,
)
fig.show()

## XL-MS crosslink validation (9GV6/9GV7, outside this project's own data)

A different dataset - cross-linked mass spec results for two AlphaFold-predicted structures, from `/mnt/d/Lab/TCRBinding/XLMS/Ecolifolded/`, reusing this notebook's own structure-plotting functions (`plot_structure_3d`, now with a `crosslinks` parameter added). Crosslinks are drawn as gold lines between the two residues' CA atoms.

Chain-number mapping (`9GV6_1`, `9GV6_4`, ... in the `*xvis.csv` files) was confirmed empirically, not assumed: positions 1-5 map to chains A-E in order, and both `9gv6.pdb`/`9gv7.pdb` only have chains A, C, D, E (no B - likely an unmodeled β2-microglobulin), matching the `*xvis_proteinLength.csv` files, which only ever reference suffixes 1, 3, 4, 5.

The structure prefix in Protein1/Protein2 (9GV6 vs 9GV7) turned out to be an annotation artifact, not a real cross-structure link - every row in `TCRDxvis.csv` is actually intra-structure for 9GV6, and every row in `TCRExvis.csv` is intra-structure for 9GV7, regardless of what prefix is written. Only the chain-number suffix is used, not the prefix.

`AbsPos` is 0-indexed, not the PDB's own 1-indexed `resseq` directly - confirmed against the actual link count (TCRD has exactly 7 distinct raw crosslink rows in the file, a hard ceiling that an earlier attempt at dropping `AbsPos==0` as "invalid" undercounted). `+1` is applied to every position to get the real residue number.

In [22]:
XLMS_DIR = "/mnt/d/Lab/TCRBinding/XLMS/Ecolifolded"
CHAIN_SUFFIX_MAP = {1: "A", 2: "B", 3: "C", 4: "D", 5: "E"}


def parse_crosslinks(xvis_csv):
    """
    Parses a *xvis.csv crosslink file (Protein1/Protein2 like "9GV6_4|Chains",
    AbsPos1/AbsPos2 = residue number) into a deduplicated list of
    ((chain1, resseq1), (chain2, resseq2)) pairs.

    The structure prefix in Protein1/Protein2 (9GV6 vs 9GV7) is an annotation artifact,
    not a real cross-structure link - every row in a given file is intra-structure for
    that file's own structure, so only the chain-number suffix is used here, not the
    prefix.

    AbsPos is 0-indexed (position 0 = the first residue of that chain) - converted to
    the PDB's own 1-indexed resseq via +1, applied to every row (not just the AbsPos==0
    ones, which looked like invalid placeholders before accounting for the indexing
    convention - they're valid links to each chain's first residue).
    """
    df = pd.read_csv(xvis_csv)

    def parse_chain_num(value):
        # "9GV6_4|Chains" -> 4 (the structure prefix before "_" is ignored - see above)
        return int(value.split("_", 1)[1].split("|")[0])

    links = set()
    for _, row in df.iterrows():
        pos1, pos2 = int(row["AbsPos1"]) + 1, int(row["AbsPos2"]) + 1
        chain1 = CHAIN_SUFFIX_MAP[parse_chain_num(row["Protein1"])]
        chain2 = CHAIN_SUFFIX_MAP[parse_chain_num(row["Protein2"])]
        links.add(tuple(sorted([(chain1, pos1), (chain2, pos2)])))

    print(f"{xvis_csv}: {len(links)} unique crosslinks")
    return list(links)


tcrd_crosslinks = parse_crosslinks(f"{XLMS_DIR}/results/TCRDxvis.csv")
tcre_crosslinks = parse_crosslinks(f"{XLMS_DIR}/results/TCRExvis.csv")

/mnt/d/Lab/TCRBinding/XLMS/Ecolifolded/results/TCRDxvis.csv: 7 unique crosslinks
/mnt/d/Lab/TCRBinding/XLMS/Ecolifolded/results/TCRExvis.csv: 9 unique crosslinks


In [23]:
# Hand-picked, maximally-distinct chain colors (the default qualitative palette's first
# two entries were both blue-ish purples, too close together for only 4 chains) - the
# same blue/red/green/purple already used elsewhere in this project's own color scheme.
XLMS_CHAIN_COLORS = {"A": "#1f77b4", "C": "#d62728", "D": "#2ca02c", "E": "#9467bd"}

plot_structure_3d(
    f"{XLMS_DIR}/structures_wlink/alphfold/9gv6.pdb", ca_only=True, color_by="chain",
    color_map=XLMS_CHAIN_COLORS, show_atoms=False,
    crosslinks=tcrd_crosslinks, crosslink_width=10, crosslink_colorscale="Oranges", crosslink_min_length=1.0, bond_opacity=0.4,
    title="9GV6 (TCRD) with XL-MS crosslinks",
).show()

In [24]:
plot_structure_3d(
    f"{XLMS_DIR}/structures_wlink/alphfold/9gv7.pdb", ca_only=True, color_by="chain",
    color_map=XLMS_CHAIN_COLORS, show_atoms=False,
    crosslinks=tcre_crosslinks, crosslink_width=10, crosslink_colorscale="Oranges", crosslink_min_length=1.0, bond_opacity=0.4,
    title="9GV7 (TCRE) with XL-MS crosslinks",
).show()

### Refined structures (post XL-MS restraint refinement)

Loads the PyRosetta-refined structures written by `src/RefineXLStructure.py` (torsion-space minimization against the same crosslinks, FlatHarmonicFunc restraint with a 25 A max CA-CA distance - see that script for the full rationale) and plots them the same way as the originals above, for a direct before/after comparison. The refinement itself isn't rerun here - it's slow enough that it only runs as its own script, and this just visualizes its output.

In [25]:
plot_structure_3d(
    f"{XLMS_DIR}/structures_wlink/alphfold/9gv6_xlrefined.pdb", ca_only=True, color_by="chain",
    color_map=XLMS_CHAIN_COLORS, show_atoms=False,
    crosslinks=tcrd_crosslinks, crosslink_width=10, crosslink_colorscale="Oranges", crosslink_min_length=1.0, bond_opacity=0.4,
    title="9GV6 (TCRD), XL-refined",
).show()

In [26]:
plot_structure_3d(
    f"{XLMS_DIR}/structures_wlink/alphfold/9gv7_xlrefined.pdb", ca_only=True, color_by="chain",
    color_map=XLMS_CHAIN_COLORS, show_atoms=False,
    crosslinks=tcre_crosslinks, crosslink_width=10, crosslink_colorscale="Oranges", crosslink_min_length=1.0, bond_opacity=0.4,
    title="9GV7 (TCRE), XL-refined",
).show()

### Comparative structures (allele-matched, crossover-flagged, most TCR-similar to each XL-MS structure)

`src/FindCrossoverStructures.py` scanned every structure in `./dat` for the same TCRa-110-127-closer-to-TCRb-Nterm "crossover" topology, then ranked the crossover hits by TCR sequence similarity to 9GV6/9GV7, restricted to the same MHC allele (HLA-A*02:01 - confirmed by exact sequence match over the first 181 residues of chain A; exact peptide matching wasn't usable as a filter, since neither 9GV6's nor 9GV7's own peptide appears anywhere in this project's data). Best matches: `Pos_217587` for 9GV6 (78.4% mean TCRa/TCRb identity), `Pos_63656` for 9GV7 (89.9%).

Plotted here with the *same* crosslink (chain, resseq) pairs as the original XL-MS structure - not realigned or remapped to this structure's own numbering. That's exact for the MHC chain (allele-matched, near-identical sequence) but only approximate for the TCR chains, since `Pos_217587`/`Pos_63656` aren't sequence-identical to 9GV6/9GV7 (especially `Pos_217587`'s TCRa, at only 61% identity) - so a given resseq doesn't necessarily land on the same structural/IMGT position. Treat this as "what would these same distances look like on a similar TCR," not a literal transplant.

In [27]:
plot_structure_3d(
    "../dat/Pos_217587/Structure/Complex.pdb", ca_only=True, color_by="chain",
    color_map=XLMS_CHAIN_COLORS, show_atoms=False,
    crosslinks=tcrd_crosslinks, crosslink_width=10, crosslink_colorscale="Oranges", crosslink_min_length=1.0, bond_opacity=0.4,
    title="Pos_217587 (comparative to 9GV6) with 9GV6's crosslinks",
).show()

Skipped 1 crosslink(s) referencing a residue not in this structure: [(('A', 213), ('A', 269))]


In [28]:
plot_structure_3d(
    "../dat/Pos_63656/Structure/Complex.pdb", ca_only=True, color_by="chain",
    color_map=XLMS_CHAIN_COLORS, show_atoms=False,
    crosslinks=tcre_crosslinks, crosslink_width=10, crosslink_colorscale="Oranges", crosslink_min_length=1.0, bond_opacity=0.4,
    title="Pos_63656 (comparative to 9GV7) with 9GV7's crosslinks",
).show()

Skipped 2 crosslink(s) referencing a residue not in this structure: [(('C', 11), ('D', 54)), (('A', 1), ('A', 265))]


### Crosslink lengths: raw XL-MS structure vs. comparative crossover structure

Same crosslink (chain, resseq) pairs, measured on both structures - the original 9GV6/9GV7 and their comparative crossover-flagged match (`Pos_217587`/`Pos_63656`). x = structure, y = crosslink length, with each individual crosslink's two points connected by a solid line so its change is easy to follow. A second, dashed line in the same color marks that crosslink's own **predicted max Cα-Cα length**: sulfo-SDA's spacer arm (~3.9 Å) plus each of the two crosslinked residues' own side-chain max reach (Cα-to-farthest-heavy-atom distance in a fully extended rotamer - derivation and a sanity check against literature lysine/arginine values are in the next cell). The flat dotted line at 25 Å is `RefineXLStructure.py`'s own generic cutoff, kept for reference since it's a single number meant to cover every residue pair, whereas the dashed per-crosslink lines are tighter and residue-specific. Crosslinks not resolvable on one of the two structures (different residue numbering - see the note above) are dropped from this comparison rather than guessed at.

#### Predicted max linked Cα-Cα length, per residue pair

`predicted_max = sulfo-SDA spacer arm (~3.9 Å) + residue1's own side-chain max reach + residue2's own side-chain max reach`. Sulfo-SDA is heterobifunctional (NHS-ester + photoreactive diazirine); the diazirine end reacts non-specifically (C-H insertion anywhere nearby, not just at a particular side chain), so the same per-residue reach is applied to both ends regardless of which one carried the amine-reactive group.

Side-chain max reach = Cα-to-farthest-heavy-atom distance in a fully extended (all-trans) rotamer: every rotatable side-chain dihedral set to 180° (ring-internal bonds - Phe/Tyr/Trp/His rings, Pro's ring - are never rotated, since real rings are rigid), bond lengths/angles from an MMFF94-optimized RDKit 3D embedding of each free amino acid - computed here, not copied from a single secondary source. Branch points (e.g. Leu's two symmetric methyls, Asp's two carboxylate oxygens) are each evaluated independently in their own maximally-extended conformation, so the reported value is always the true per-residue maximum, not an average or a single arbitrarily-picked branch.

Sanity check against a sulfo-SDA crosslinker patent's own cited side-chain lengths: this computation gives Lys 6.36 Å (patent: ~6.3 Å) and Arg 7.39 Å (patent: ~7.1 Å) - close enough to trust the method for the other 18 residues, which aren't independently cited anywhere we found. Like the 25 Å line above, this is an upper bound on what the crosslinker chemistry can physically reach, not a target or expected distance - a measured length under this line is plausible, not confirmed; a length over it is a real problem for that structure, tighter evidence than the generic 25 Å cutoff gives.

In [ ]:
# Sulfo-SDA spacer-arm length (reactive-group-to-reactive-group) - see RefineXLStructure.py's
# own note on sulfo-SDA and MAX_DISTANCE. ~3.9 A is the commonly-cited figure for sulfo-SDA.
SDA_SPACER_A = 3.9

# Ca-to-farthest-heavy-atom distance per residue, fully extended (all-trans) rotamer -
# computed once with RDKit (ETKDGv3 embed + MMFF94 optimize + every rotatable side-chain
# dihedral set to 180 deg, ring-internal bonds untouched); see the markdown above for the
# method and its sanity check against literature Lys/Arg values.
SIDECHAIN_REACH_A = {
    "A": 1.52, "R": 7.39, "N": 3.74, "D": 3.71, "C": 2.82, "Q": 4.97, "E": 4.97, "G": 0.00,
    "H": 4.64, "I": 3.97, "L": 3.99, "K": 6.36, "M": 5.32, "F": 5.17, "P": 2.43, "S": 2.44,
    "T": 2.58, "W": 6.72, "Y": 6.45, "V": 2.55,
}
THREE_TO_ONE = {
    "ALA": "A", "ARG": "R", "ASN": "N", "ASP": "D", "CYS": "C", "GLN": "Q", "GLU": "E",
    "GLY": "G", "HIS": "H", "ILE": "I", "LEU": "L", "LYS": "K", "MET": "M", "PHE": "F",
    "PRO": "P", "SER": "S", "THR": "T", "TRP": "W", "TYR": "Y", "VAL": "V",
}


def predicted_max_crosslink(resname1, resname2):
    """Predicted max Ca-Ca distance (A) sulfo-SDA can plausibly bridge for this specific
    crosslinked residue pair - see the markdown above."""
    return SIDECHAIN_REACH_A[THREE_TO_ONE[resname1]] + SDA_SPACER_A + SIDECHAIN_REACH_A[THREE_TO_ONE[resname2]]

In [ ]:
def crosslink_lengths(pdb_path, crosslinks):
    """One row per resolvable crosslink: chain1, resseq1, resname1, chain2, resseq2,
    resname2, length_A - pairs referencing a residue not present in this structure are
    skipped, not raised."""
    atoms = load_structure_atoms(pdb_path, ca_only=True)
    ca_pos = {(r.chain, r.resseq): np.array([r.x, r.y, r.z]) for r in atoms.itertuples()}
    resname = {(r.chain, r.resseq): r.resname for r in atoms.itertuples()}
    rows = []
    for (c1, r1), (c2, r2) in crosslinks:
        if (c1, r1) not in ca_pos or (c2, r2) not in ca_pos:
            continue
        length = float(np.linalg.norm(ca_pos[(c1, r1)] - ca_pos[(c2, r2)]))
        rows.append({"chain1": c1, "resseq1": r1, "resname1": resname[(c1, r1)],
                     "chain2": c2, "resseq2": r2, "resname2": resname[(c2, r2)], "length_A": length})
    return pd.DataFrame(rows)


def compare_crosslink_lengths(xlms_label, raw_pdb, crossover_pdb, crosslinks):
    raw = crosslink_lengths(raw_pdb, crosslinks)
    crossover = crosslink_lengths(crossover_pdb, crosslinks)
    merged = raw.merge(crossover, on=["chain1", "resseq1", "chain2", "resseq2"], suffixes=("_raw", "_crossover"))
    merged["XLMS_structure"] = xlms_label
    merged["label"] = merged.apply(lambda r: f"{r.chain1}{r.resseq1}-{r.chain2}{r.resseq2}", axis=1)
    # Predicted max uses the raw XL-MS structure's own residue identity at each site - the
    # comparative crossover structure has a different sequence, but the crosslink chemistry
    # (which residue the mass spec actually identified) is a property of the raw structure.
    merged["predicted_max_CA_CA_A"] = merged.apply(
        lambda r: predicted_max_crosslink(r.resname1_raw, r.resname2_raw), axis=1)
    return merged


length_comparison = pd.concat([
    compare_crosslink_lengths(
        "9GV6", f"{XLMS_DIR}/structures_wlink/alphfold/9gv6.pdb",
        "../dat/Pos_217587/Structure/Complex.pdb", tcrd_crosslinks,
    ),
    compare_crosslink_lengths(
        "9GV7", f"{XLMS_DIR}/structures_wlink/alphfold/9gv7.pdb",
        "../dat/Pos_63656/Structure/Complex.pdb", tcre_crosslinks,
    ),
], ignore_index=True)
length_comparison

In [ ]:
STRUCTURE_ORDER = ["9GV6", "Pos_217587", "9GV7", "Pos_63656"]
STRUCTURE_NAMES = {
    ("9GV6", "length_A_raw"): "9GV6", ("9GV6", "length_A_crossover"): "Pos_217587",
    ("9GV7", "length_A_raw"): "9GV7", ("9GV7", "length_A_crossover"): "Pos_63656",
}
XLMS_GROUP_COLORS = {"9GV6": "#1f77b4", "9GV7": "#d62728"}

long_df = length_comparison.melt(
    id_vars=["XLMS_structure", "label", "predicted_max_CA_CA_A"],
    value_vars=["length_A_raw", "length_A_crossover"],
    var_name="RawOrComparative", value_name="length_A",
)
long_df["Structure"] = long_df.apply(lambda r: STRUCTURE_NAMES[(r.XLMS_structure, r.RawOrComparative)], axis=1)

fig = go.Figure()
for xlms_label, color in XLMS_GROUP_COLORS.items():
    group = long_df[long_df["XLMS_structure"] == xlms_label]
    for crosslink_label, cl_group in group.groupby("label"):
        cl_group = cl_group.set_index("Structure").reindex(
            [s for s in STRUCTURE_ORDER if s in cl_group["Structure"].values]
        )
        fig.add_trace(go.Scatter(
            x=cl_group.index, y=cl_group["length_A"],
            mode="lines+markers",
            line=dict(color=color, width=1),
            marker=dict(color=color, size=10, line=dict(color="black", width=0.5)),
            name=f"{xlms_label} {crosslink_label}", legendgroup=xlms_label, showlegend=False,
            hovertext=f"{xlms_label} {crosslink_label}", hoverinfo="y+text",
        ))
        # Predicted max for this specific crosslink (sulfo-SDA spacer + each residue's own
        # side-chain reach) - flat across structures, since it's a property of which residues
        # were crosslinked, not of the 3D structure being measured.
        pred = cl_group["predicted_max_CA_CA_A"].iloc[0]
        fig.add_trace(go.Scatter(
            x=cl_group.index, y=[pred] * len(cl_group),
            mode="lines", line=dict(color=color, width=1.5, dash="dash"), opacity=0.5,
            name=f"{xlms_label} {crosslink_label} predicted max", legendgroup=f"{xlms_label}_pred",
            showlegend=False, hovertext=f"{xlms_label} {crosslink_label} predicted max: {pred:.1f} A",
            hoverinfo="y+text",
        ))

# Dummy traces purely so the legend shows one entry per XL-MS structure group (plus one for
# the predicted-max lines), not one per individual crosslink line.
for xlms_label, color in XLMS_GROUP_COLORS.items():
    fig.add_trace(go.Scatter(
        x=[None], y=[None], mode="markers", marker=dict(color=color, size=10), name=xlms_label,
    ))
fig.add_trace(go.Scatter(
    x=[None], y=[None], mode="lines", line=dict(color="black", width=1.5, dash="dash"),
    name="Predicted max (sulfo-SDA + per-residue reach)",
))

fig.add_hline(y=25, line_dash="dot", line_color="black", opacity=0.6)
fig.update_layout(
    title="Crosslink length by structure",
    xaxis=dict(categoryorder="array", categoryarray=STRUCTURE_ORDER, title="Structure"),
    yaxis_title="Crosslink length (A)",
    width=800, height=600,
)
fig.show()

## 9GV7 only, within-TCR crosslinks

Same point/slope comparison as above, but narrowed two ways: only the 9GV7 side (9GV7 raw vs its Pos_63656 comparative crossover structure), and only crosslinks with both ends on a TCR chain (D=TCRa or E=TCRb) - excluding any crosslink that touches the MHC (A) or peptide (C) chain, to isolate evidence about the TCR ectodomain's own conformation specifically.

In [ ]:
tcr_only_9gv7 = length_comparison[
    (length_comparison["XLMS_structure"] == "9GV7")
    & length_comparison["chain1"].isin(["D", "E"])
    & length_comparison["chain2"].isin(["D", "E"])
]
n_total_9gv7 = (length_comparison["XLMS_structure"] == "9GV7").sum()
print(f"{len(tcr_only_9gv7)} of {n_total_9gv7} 9GV7 crosslinks are within-TCR (both ends on chain D or E)")

STRUCTURE_ORDER_9GV7 = ["9GV7", "Pos_63656"]
color = XLMS_GROUP_COLORS["9GV7"]

long_df_9gv7 = tcr_only_9gv7.melt(
    id_vars=["XLMS_structure", "label", "predicted_max_CA_CA_A"],
    value_vars=["length_A_raw", "length_A_crossover"],
    var_name="RawOrComparative", value_name="length_A",
)
long_df_9gv7["Structure"] = long_df_9gv7["RawOrComparative"].map({
    "length_A_raw": "9GV7", "length_A_crossover": "Pos_63656",
})

fig = go.Figure()
for crosslink_label, cl_group in long_df_9gv7.groupby("label"):
    cl_group = cl_group.set_index("Structure").reindex(STRUCTURE_ORDER_9GV7)
    fig.add_trace(go.Scatter(
        x=cl_group.index, y=cl_group["length_A"],
        mode="lines+markers",
        line=dict(color=color, width=1),
        marker=dict(color=color, size=10, line=dict(color="black", width=0.5)),
        name=crosslink_label, showlegend=True,
        hovertext=crosslink_label, hoverinfo="y+text",
    ))
    pred = cl_group["predicted_max_CA_CA_A"].iloc[0]
    fig.add_trace(go.Scatter(
        x=cl_group.index, y=[pred] * len(cl_group),
        mode="lines", line=dict(color=color, width=1.5, dash="dash"), opacity=0.5,
        name=f"{crosslink_label} predicted max", showlegend=False,
        hovertext=f"{crosslink_label} predicted max: {pred:.1f} A", hoverinfo="y+text",
    ))

fig.add_trace(go.Scatter(
    x=[None], y=[None], mode="lines", line=dict(color="black", width=1.5, dash="dash"),
    name="Predicted max (sulfo-SDA + per-residue reach)",
))
fig.add_hline(y=25, line_dash="dot", line_color="black", opacity=0.6)
fig.update_layout(
    title="9GV7: within-TCR crosslink length by structure",
    xaxis=dict(categoryorder="array", categoryarray=STRUCTURE_ORDER_9GV7, title="Structure"),
    yaxis_title="Crosslink length (A)",
    width=600, height=600,
)
fig.show()